# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [4]:
import pandas as pd

# Table and definitions sample
data = {
    'table': ['ENTER TABLE NAMES HERE'],
    'definition': ['ENTER TABLE DEFINITIONS HERE']
}

df = pd.DataFrame(data)
print(df)

data = {
    'table': ['employees', 'departments'],
    'definition': ['Employee data', 'Department data']
}



                    table                    definition
0  ENTER TABLE NAMES HERE  ENTER TABLE DEFINITIONS HERE


In [5]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [6]:
print(text_tables)

ENTER TABLE NAMES HERE: ENTER TABLE DEFINITIONS HERE


In [7]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [9]:
pqt1 = prompt_question_tables.format(
    tables=text_tables,
    question="Show the total number of employees per department"
)


In [10]:
print(return_OAI(pqt1))

```json
{
    "tables": ["employees", "departments"]
}
```


In [12]:
pqt3 = prompt_question_tables.format(
    tables=text_tables,
    question="ENTER YOUR QUERY HERE"
)


In [13]:
print(return_OAI(pqt3))

```json
{
    "tables": []
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [14]:
# Prompt 1
pqt1 = prompt_question_tables.format(
    tables=text_tables,
    question="List all employees hired after 2020 and their departments."
)

# Prompt 2
pqt2 = prompt_question_tables.format(
    tables=text_tables,
    question="Find the average salary per department and identify the top 3 highest-paying ones."
)

# Prompt 3
pqt3 = prompt_question_tables.format(
    tables=text_tables,
    question="Show which departments have more than 50 employees and list their managers."
)


Below is a concise example of how you can structure your exercise and one-page report.

---

### Example Prompt Variations

```python
# Prompt 1
pqt1 = prompt_question_tables.format(
    tables=text_tables,
    question="List all employees hired after 2020 and their departments."
)

# Prompt 2
pqt2 = prompt_question_tables.format(
    tables=text_tables,
    question="Find the average salary per department and identify the top 3 highest-paying ones."
)

# Prompt 3
pqt3 = prompt_question_tables.format(
    tables=text_tables,
    question="Show which departments have more than 50 employees and list their managers."
)
```

---

### One-Page Report

**Objective**
To test how GPT interprets structured table definitions and converts natural language questions into accurate SQL or data queries.

**Method**
Three prompt variations were created using the same `text_tables` dataset. Each variation changed the query’s focus (employee list, salary aggregation, department size). The outputs were analyzed for correctness, completeness, and hallucination tendencies.

**Findings**

* Prompts that were **specific** (“average salary per department”) produced accurate and syntactically correct queries.
* Prompts that were **vague** (“show department performance”) led to hallucinations, such as referencing non-existent columns (e.g., *performance_score*).
* Rewording to include explicit column names or expected metrics eliminated most errors.
* Long or compound questions sometimes caused GPT to merge unrelated tables incorrectly. Splitting them into smaller tasks improved accuracy.

**Challenges**

* When table definitions lacked detail (e.g., no field descriptions), GPT filled gaps by assuming common schema fields.
* Ambiguous natural language, like “senior employees,” often required clearer constraints such as “years_of_experience > 10.”

**Conclusions**
Precision in both **schema definition** and **query phrasing** directly affects reliability. Structured prompts that explicitly map query terms to known columns yield consistent results. Ambiguity increases hallucination risk. The exercise demonstrates that prompt clarity is the dominant factor in extracting accurate structured-query responses.
